# John Tree
Using the names, origins, and spellings on behindthename.com's tree of related names under the name John, I want to find the number of babies with each name in the United States and organize a new graph - the American John Tree. Names were recorded and organized manually using a depth first search of the tree, recording a category for how that name was attained from the parent name.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import functions as fun

In [2]:
john_tree = pd.read_csv("Name Investigations - John Tree.csv")
john_tree.head()

,John Tree Names,Gender,Languages,Form,Parent Name,Level
0,Yahweh,M,Hebrew,Full,NaN,0
1,Yehochanan,M,Hebrew,Full,Yahweh,1
2,Jehohanan,M,Hebrew,Full,Yehochanan,2
3,Yochanan,M,Hebrew,Diminutive,Yehochanan,2
4,Hovhannes,M,Armenian,Full,Yochanan,3


In [3]:
class name_node:
    name = ""
    versions = [] # in form of tuple (level, gender, language, form)
    ins = [] # parent nodes
    outs = [] # child nodes
    sums = (0, 0) # as a tuple of (female sum, male sum)
    peak_years = (1880, 1880) # as a tuple of (female sum, male sum)
    peak_sums = (0, 0) # as a tuple of (female sum, male sum)

    def __init__(self, name, parent):
        self.name = name
        self.parent = parent
    def add_version(self, version):
        self.versions.append(version)
    def add_parent(self, parent):
        self.ins.append(parent)
    def add_child(self, child):
        self.outs.append(child)
    def add_sums(self, f, m):
        self.sums = (f, m)
    def add_peak_years(self, f, m):
        self.peak_years = (f, m)
    def add_peaks(self, f, m):
        self.peak_sums = (f, m)


In [4]:
Yahweh = name_node("Yahweh", "") # root is the name Yahweh
jt = {"Yahweh" : Yahweh}
for i in range(1, len(john_tree)):
    name = john_tree['John Tree Names'][i]
    parent = john_tree['Parent Name'][i]
    curr_node = ""
    if name not in jt:
        curr_node = name_node(name, jt[parent])
    else:
        curr_node = jt[name]
        if parent not in curr_node.ins:
            jt[name].add_parent(jt[parent])
    curr_node.add_version((john_tree['Level'], john_tree['Gender'], 
                          john_tree['Languages'], john_tree['Form']))
    jt[name] = curr_node
    if name not in jt[parent].outs:
        jt[parent].add_child(curr_node)
    
    

In [5]:
fun.init(1880, 2025, "names")

In [6]:
for n in list(jt.keys()):
    f_count, _ = fun.name_counts_years(n, 'f')
    m_count, _ = fun.name_counts_years(n, 'm')
    curr_node = jt[n]
    curr_node.add_sums(sum(f_count), sum(m_count))
    f_year, f_peak = fun.peak_year(f_count)
    m_year, m_peak = fun.peak_year(m_count)
    curr_node.add_peak_years(f_year, m_year)
    curr_node.add_peaks(f_peak, m_peak)
    jt[n] = curr_node
